# AC-MOT v10 — FULL VisDrone MOT test-dev (17 sequences)

Independent full-dataset run. Original notebook and prior results are preserved.

In [ ]:
from pathlib import Path
import json, runpy, shutil, gc
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
# Load setup/modules from the preserved original notebook, without running its old result cells.
import subprocess, sys
nb_path = Path('/content/AC_MOT_v10_original.ipynb')
if not nb_path.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    subprocess.run([sys.executable, '-m', 'gdown', '--id', '1UfmcfhluB3zd_61V5isTdLRugjG1twXY', '-O', str(nb_path), '--fuzzy'], check=True)
nb = json.loads(nb_path.read_text())
code_cells = [c for c in nb['cells'] if c.get('cell_type') == 'code']
for idx, cell in enumerate(code_cells[:3]):
    exec(compile(''.join(cell['source']), f'AC_MOT_v10_original_cell_{idx+1}', 'exec'), globals())
# Force the completed 17-sequence dataset.
DATA_ROOT = Path('/content/drive/MyDrive/VisDrone2019-MOT-test-dev')
SEQ_DIR = DATA_ROOT / 'sequences'
ANNOT_DIR = DATA_ROOT / 'annotations'
VAL_SEQS_NAMES = sorted([p.name for p in SEQ_DIR.iterdir() if p.is_dir() and any(p.glob('*.jpg'))])
by_name = {s.name: s for s in all_sequences}
VAL_SEQS = [by_name[n] for n in VAL_SEQS_NAMES if n in by_name]
print(f'FULL DATASET CHECK: {len(VAL_SEQS)} sequences, {sum(len(list(s.glob("*.jpg"))) for s in VAL_SEQS)} frames, {sum((ANNOT_DIR/f"{s.name}.txt").exists() for s in VAL_SEQS)} annotations')
# Fresh main run (3 production systems × 17 sequences).
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
run_tag = f'acmot_v10_full17_{ts}'
all_rows = [run_system(system, VAL_SEQS, run_tag) for system in SYSTEMS]
main_df = pd.concat(all_rows, ignore_index=True)
main_summary = summarize(main_df)
summary_path = DRIVE_RESULTS / f'{run_tag}_summary.csv'
seq_path = DRIVE_RESULTS / f'{run_tag}_per_sequence.csv'
main_summary.to_csv(summary_path, index=False)
main_df.to_csv(seq_path, index=False)
print(main_summary[['system','sequences','mota','idf1','recall','hota','ids','fps','fn','fp','mean_imgsz']].to_string(index=False))
print(f'Saved -> {summary_path}')
print(f'Saved -> {seq_path}')
